# Introduction to Vector Databases

Once documents are chunked and embedded, you need a system to **store millions of vectors** and answer **"which chunks are closest to this query?"** thousands of times per day. A **vector database** (or vector index) is specialized storage for high-dimensional embeddings plus **metadata**, **IDs**, and optional **document text**.

You can prototype with NumPy lists; production RAG needs **persistence**, **filtered search**, **upserts**, and **approximate nearest neighbor (ANN)** indexes for speed. This notebook explains the data model, core operations, exact vs ANN search, how vector DBs differ from SQL, an in-memory reference implementation, and a retrieval demo with real OpenAI embeddings.

Prerequisites: **01–04** in this section (especially similarity metrics and chunking).

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)

---

## 1. What Is a Vector Database?

A vector database optimizes three concerns together:

1. **Store** — vectors + payload (text, metadata)
2. **Search** — k-nearest neighbors by cosine / L2 / dot product
3. **Filter** — restrict search by metadata (`source`, `tenant_id`)

```
         ┌─────────────────────────────┐
Ingest → │  Collection: docs_prod_v1   │
         │  id | vector | doc | meta  │
         └──────────────┬──────────────┘
                        │ query vector + optional filters
                        ▼
                   top-k hits
```

### 1.1 When NumPy Is Enough vs When You Need a Vector DB

| Situation | Approach |
|-----------|----------|
| Notebook demo, < 5k vectors | NumPy brute force |
| Prototype RAG, local dev | Chroma in-memory or persistent |
| Production, millions of vectors | Managed or self-hosted vector DB + ANN |
| Already on PostgreSQL | pgvector extension |

The **API shape** (upsert, query, delete) is similar across tools — learn the concepts here, then map to Chroma (notebook 06).

---

## 2. Data Model

### 2.1 Record Fields

| Field | Type | Purpose |
|-------|------|---------|
| **id** | string (unique) | Upsert key, citations, deletes |
| **vector** | `float[d]` | Embedding; fixed $d$ per collection |
| **document** | string (optional) | Original chunk text for LLM context |
| **metadata** | JSON object | Filters, source path, section, version |

Example record:

```python
{
    "id": "api-docs#migrations#0",
    "vector": [0.012, -0.034, ...],  # len = 1536
    "document": "Alembic manages schema migrations...",
    "metadata": {"source": "api-docs.md", "section": "Database Migrations"},
}
```

### 2.2 Collections (Indexes)

A **collection** groups vectors that share:

- **Dimension** $d$ (must match embedding model)
- **Distance metric** (cosine, L2, inner product)
- **Embedding model version** (convention — enforce in app code)

Use separate collections for **dev vs prod**, or **index versions** (`docs_v1`, `docs_v2`) during migrations.

---

## 3. Core Operations

| Operation | Description | RAG usage |
|-----------|-------------|----------|
| **Create collection** | Define name, metric, dimension | Once per index version |
| **Upsert** | Insert or replace by `id` | Ingest / re-index chunks |
| **Query** | Top-k by similarity to query vector | Every user search |
| **Get** | Fetch by id(s) | Debugging, parent–child fetch |
| **Delete** | Remove ids or by metadata filter | Stale doc removal |
| **Update metadata** | Change filters without re-embed | Doc classification fixes |

### 3.1 Upsert Semantics

**Upsert** = update if `id` exists, insert otherwise. Enables **idempotent ingest**: re-run the pipeline without duplicating rows.

### 3.2 Query Semantics

Typical query inputs:

- `query_vector` **or** `query_text` (if DB embeds for you)
- `n_results` = k
- `where` = metadata filter (notebook 07)
- `include` = documents, metadatas, distances

### 3.3 Re-Indexing Rule

Changing **embedding model** or **dimensions** invalidates stored vectors. Create a **new collection**, re-embed all chunks, evaluate, then cut traffic over.

---

## 4. Exact Search vs Approximate Nearest Neighbor (ANN)

### 4.1 Exact k-NN

Compare query to **every** vector. Recall = 100%. Cost $O(N \cdot d)$ per query.

### 4.2 ANN Indexes

At millions of vectors, exact search is too slow. **ANN** algorithms return **approximate** top-k in sub-linear time.

| Algorithm | Idea | Trade-off |
|-----------|------|----------|
| **HNSW** | Navigable small-world graph | Fast, popular default |
| **IVF** | Cluster partitions; search nearby clusters | Tunable `nprobe` |
| **PQ / compression** | Smaller memory footprint | Some accuracy loss |

```
Exact:  scan all N        → 100% recall, slow
ANN:    visit << N nodes  → 95–99% recall typical, fast
```

Tune ANN when notebook **09** shows retrieval misses at scale — not on day one of prototyping.

---

## 5. Vector DB vs Relational DB vs Search Engine

| System | Strength | Weakness for semantic RAG |
|--------|----------|---------------------------|
| **Vector DB** | Similarity search in $\mathbb{R}^d$ | Not a full SQL replacement |
| **PostgreSQL** | Transactions, joins, constraints | No native ANN until **pgvector** |
| **Elasticsearch** | BM25 keyword + hybrid plugins | Heavier ops stack |
| **NumPy list** | Simple | No persistence, no ANN |

**pgvector** — store vectors in a Postgres column; one database for app data + embeddings. Good when you already run Postgres and scale is moderate.

**Hybrid RAG** — combine vector hits with BM25 keyword hits; merge rankings (notebook 08).

---

## 6. Persistence and Deployment Modes

| Mode | Behavior | Use |
|------|----------|-----|
| **In-memory** | Lost on process exit | Tests, quick demos |
| **Embedded file** | Local directory (Chroma PersistentClient) | Dev laptops |
| **Server** | Dedicated vector DB service | Production |
| **Managed cloud** | Vendor-hosted | Low ops, pay per use |

Always back up collection data and document **which embedding model** built the index.

---

## 7. Reference Implementation: In-Memory Vector Store

The class below mirrors what production DBs do — without ANN or disk. Use it to understand upsert, cosine query, get, delete, and metadata filtering.

In [ ]:
from dataclasses import dataclass, field
from typing import Any


@dataclass
class VectorRecord:
    id: str
    vector: np.ndarray
    document: str
    metadata: dict[str, Any] = field(default_factory=dict)


class InMemoryVectorStore:
    """Minimal vector store — exact cosine search, metadata filter."""

    def __init__(self, metric: str = "cosine"):
        self.metric = metric
        self._records: dict[str, VectorRecord] = {}

    def upsert(self, records: list[VectorRecord]) -> int:
        for r in records:
            self._records[r.id] = r
        return len(records)

    def get(self, ids: list[str]) -> list[VectorRecord]:
        return [self._records[i] for i in ids if i in self._records]

    def delete(self, ids: list[str]) -> int:
        n = 0
        for i in ids:
            if self._records.pop(i, None):
                n += 1
        return n

    def count(self) -> int:
        return len(self._records)

    def _cosine(self, a: np.ndarray, b: np.ndarray) -> float:
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

    def query(
        self,
        query_vector: np.ndarray,
        k: int = 3,
        where: dict[str, Any] | None = None,
    ) -> list[tuple[VectorRecord, float]]:
        candidates = self._records.values()
        if where:
            candidates = [
                r for r in candidates
                if all(r.metadata.get(key) == val for key, val in where.items())
            ]
        scored = [(r, self._cosine(r.vector, query_vector)) for r in candidates]
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:k]


print("InMemoryVectorStore defined.")

---

## 8. Demonstration: Toy 2D Vectors

In [ ]:
store = InMemoryVectorStore()
store.upsert([
    VectorRecord("auth", np.array([0.95, 0.05]), "JWT authentication guide", {"topic": "security"}),
    VectorRecord("jwt",  np.array([0.90, 0.10]), "Bearer token format", {"topic": "security"}),
    VectorRecord("db",   np.array([0.05, 0.95]), "Alembic migrations", {"topic": "database"}),
    VectorRecord("api",  np.array([0.85, 0.15]), "FastAPI REST endpoints", {"topic": "api"}),
])

q = np.array([0.92, 0.08])
print("Top-3 (unfiltered):")
for rec, score in store.query(q, k=3):
    print(f"  {score:.4f}  id={rec.id:4s}  topic={rec.metadata['topic']}")

print("\nTop-3 (topic=database only):")
for rec, score in store.query(q, k=3, where={"topic": "database"}):
    print(f"  {score:.4f}  id={rec.id}")

Filtered search may return **low** cosine scores — the best match within the filter, not globally.

---

## 9. Demonstration: Upsert, Get, Delete

In [ ]:
print("Count before:", store.count())

# Upsert replaces same id
store.upsert([
    VectorRecord("db", np.array([0.1, 0.9]), "Alembic upgrade head in CI", {"topic": "database"}),
])
print("After upsert db:", store.get(["db"])[0].document)

store.delete(["api"])
print("Count after delete api:", store.count())
print("IDs left:", sorted(store._records.keys()))

---

## 10. Demonstration: Real Embeddings in Memory Store

Same store class with OpenAI vectors — mirrors ingest → query flow before Chroma.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)
EMBED_MODEL = "text-embedding-3-small"

CHUNKS = [
    {"id": "c1", "text": "Alembic manages SQLAlchemy schema migrations.", "source": "db-docs"},
    {"id": "c2", "text": "JWT tokens authenticate stateless API requests.", "source": "security-docs"},
    {"id": "c3", "text": "FastAPI provides dependency injection for routes.", "source": "api-docs"},
    {"id": "c4", "text": "Pytest fixtures share database setup in tests.", "source": "test-docs"},
]


def embed(texts: list[str]) -> np.ndarray:
    r = client.embeddings.create(model=EMBED_MODEL, input=texts)
    ordered = sorted(r.data, key=lambda x: x.index)
    return np.array([d.embedding for d in ordered])


rag_store = InMemoryVectorStore()
texts = [c["text"] for c in CHUNKS]
vectors = embed(texts)
records = [
    VectorRecord(c["id"], vectors[i], c["text"], {"source": c["source"]})
    for i, c in enumerate(CHUNKS)
]
rag_store.upsert(records)
print(f"Ingested {rag_store.count()} chunks, dim={vectors.shape[1]}")

In [ ]:
user_query = "How do I version database tables?"
q_vec = embed([user_query])[0]

print(f"Query: {user_query}\n")
for rec, score in rag_store.query(q_vec, k=2):
    print(f"{score:.4f}  [{rec.metadata['source']}] {rec.document}")

---

## 11. RAG Architecture Placement

```
  Documents
      → chunk (nb 04)
      → embed (nb 03)
      → vector DB upsert (this notebook / nb 06)
  User question
      → embed query
      → vector DB query → top-k chunks
      → LLM prompt with context (08. RAG Fundamentals)
```

The vector DB is the **retrieval layer** — it does not call the LLM.

---

## 12. Scaling Checklist

| Signal | Action |
|--------|--------|
| Query latency grows with corpus | Enable ANN / upgrade plan |
| RAM pressure | Compression, shard collections |
| Multi-tenant | Separate collections or `tenant_id` filter |
| Re-index often | Batch upsert + blue/green collections |
| Need SQL + vectors | Evaluate pgvector |

---

## 13. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Mixed embedding dims in one collection | Index errors | Validate on ingest |
| No stable ids | Duplicates on re-ingest | Content-hash or chunk_id |
| Store vectors only, no document text | Extra DB round trips | Store `document` field |
| In-memory in production | Data loss on restart | Persistent client |
| Exact search at huge N | Timeouts | ANN index |
| Wrong distance metric | Bad rankings | Match model docs |

---

## 14. Summary

Vector databases store **id + vector + document + metadata** in **collections** and answer **top-k similarity queries**, often with **metadata filters** and **ANN** indexes at scale.

Core operations: **upsert**, **query**, **get**, **delete**. Re-index when the embedding model changes. NumPy and in-memory stores teach the logic; **Chroma** (next notebook) adds persistence and production APIs.

Demonstrations built a full `InMemoryVectorStore`, ran 2D and OpenAI embedding queries, and placed the vector DB in the RAG pipeline.

Next: **06. Chroma — Collections, Ingest, and Query**.